# StructParameter

A `StructParameter` returns structured data — a Python `dataclass` or Pydantic v2 `BaseModel` —
and automatically unpacks each field into a separate dataset column when used in a
`Measurement`.

This is useful when a single instrument query returns multiple related values
(e.g. voltage *and* current, or magnitude *and* phase) that belong together logically
but should be stored as individual columns in the dataset.

Key properties:

- **Get-only** — `StructParameter` is a subclass of `ParameterBase` with no set support.
- **Automatic field introspection** — field names, types, labels, and units are derived from the struct definition.
- **Dataset integration** — `Measurement.register_parameter` registers each field as a separate dependent parameter.
- **Pydantic support is optional** — works with plain dataclasses out of the box; Pydantic v2 is supported when installed.

## Setup

In [1]:
import tempfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np

from qcodes.dataset import (
    Measurement,
    initialise_or_create_database_at,
    load_or_create_experiment,
)
from qcodes.parameters import ManualParameter, StructParameter
from qcodes.station import Station

In [2]:
# Create a temporary database for this example
db_path = Path(tempfile.gettempdir()) / "struct_parameter_example.db"
initialise_or_create_database_at(str(db_path))
exp = load_or_create_experiment("struct_parameter_tutorial", sample_name="no sample")
station = Station()
rng = np.random.default_rng(42)

## Defining a struct type

The struct type is a plain Python `dataclass` (or a Pydantic `BaseModel`) whose fields
describe the shape of the data returned by the parameter.

In [3]:
@dataclass
class IVResult:
    """Result of a simultaneous I-V measurement."""

    voltage: float
    current: float

## Creating a StructParameter

To create a `StructParameter` you subclass it and implement `get_raw`.
The `struct_type` argument tells the parameter which dataclass to expect.

In [4]:
class SimulatedIV(StructParameter[IVResult, None]):
    """Simulated I-V measurement that returns voltage and current."""

    def get_raw(self) -> IVResult:
        # In a real driver this would query the instrument
        v = rng.uniform(0.0, 1.0)
        i = v / 1000.0 + rng.normal(0, 1e-5)
        return IVResult(voltage=v, current=i)

In [5]:
iv = SimulatedIV(
    "iv",
    struct_type=IVResult,
    field_labels={"voltage": "Voltage", "current": "Current"},
    field_units={"voltage": "V", "current": "A"},
)

Calling `get()` returns the full dataclass instance:

In [6]:
result = iv.get()
print(result)
print(f"  voltage = {result.voltage:.4f} V")
print(f"  current = {result.current:.6f} A")

IVResult(voltage=0.7739560485559633, current=0.0007635562074935583)
  voltage = 0.7740 V
  current = 0.000764 A


## Inspecting the parameter

The parameter exposes metadata about its fields:

In [7]:
print("names: ", iv.names)
print("labels:", iv.labels)
print("units: ", iv.units)
print("full_names:", iv.full_names)

names:  ('voltage', 'current')
labels: ('Voltage', 'Current')
units:  ('V', 'A')
full_names: ('iv_voltage', 'iv_current')


Each field has a synthetic child parameter that carries its own label, unit, and paramtype.
These are accessible via `field_parameters`:

In [8]:
for name, fp in iv.field_parameters.items():
    print(
        f"  {name}: register_name={fp.register_name!r}, "
        f"label={fp.label!r}, unit={fp.unit!r}"
    )

  voltage: register_name='iv_voltage', label='Voltage', unit='V'
  current: register_name='iv_current', label='Current', unit='A'


## Using StructParameter in a Measurement

When you register a `StructParameter` with a `Measurement`, each field is registered
as a separate dependent parameter. The struct parameter itself is *not* stored in the
dataset — only the individual fields are.

When adding results, you pass the struct parameter and the dataclass instance.
The `Measurement` automatically unpacks the struct into its field values.

In [9]:
# Create a setpoint parameter
gate = ManualParameter("gate", label="Gate Voltage", unit="V")
station.add_component(gate)

'gate'

In [10]:
meas = Measurement(exp=exp, name="iv_sweep")
meas.register_parameter(gate)
meas.register_parameter(iv, setpoints=(gate,))

with meas.run() as datasaver:
    for v_gate in np.linspace(0, 1, 20):
        gate.set(v_gate)
        iv_result = iv.get()
        datasaver.add_result(
            (gate, v_gate),
            (iv, iv_result),  # type: ignore[arg-type]
        )

dataset = datasaver.dataset
print(f"Run ID: {dataset.run_id}")
print(f"Parameters in dataset: {[p.name for p in dataset.get_parameters()]}")

Starting experimental run with id: 2. 
Run ID: 2
Parameters in dataset: ['gate', 'iv_voltage', 'iv_current']


Notice that the dataset contains `gate`, `iv_voltage`, and `iv_current` — the struct
has been unpacked into individual columns.

In [11]:
df = dataset.to_pandas_dataframe()
df.head()

,iv_current,iv_voltage
gate,,
0.000000,0.000868,0.858598
0.052632,0.000081,0.094177
0.105263,0.000758,0.761140
0.157895,0.000120,0.128114
0.210526,0.000379,0.370798


## Customising field labels, units, and paramtypes

You can customise each field's label, unit, and paramtype via
`field_labels`, `field_units`, and `field_paramtypes` arguments.

Type inference is automatic:
- `float`, `int`, `bool` → `"numeric"`
- `str` → `"text"`
- `complex` → `"complex"`
- `numpy.ndarray` → `"array"`

You can override the inferred type using `field_paramtypes`.

In [12]:
@dataclass
class ScopeTrace:
    """Oscilloscope trace with metadata."""

    waveform: np.ndarray
    sample_rate: float
    channel_name: str


class SimulatedScope(StructParameter[ScopeTrace, None]):
    def get_raw(self) -> ScopeTrace:
        t = np.linspace(0, 1e-3, 1000)
        signal = np.sin(2 * np.pi * 1000 * t) + rng.normal(0, 0.1, len(t))
        return ScopeTrace(
            waveform=signal,
            sample_rate=1e6,
            channel_name="CH1",
        )


scope = SimulatedScope(
    "scope",
    struct_type=ScopeTrace,
    field_labels={
        "waveform": "Signal",
        "sample_rate": "Sample Rate",
        "channel_name": "Channel",
    },
    field_units={"waveform": "V", "sample_rate": "Sa/s"},
)

trace = scope.get()
print(f"Channel: {trace.channel_name}")
print(f"Sample rate: {trace.sample_rate:.0f} Sa/s")
print(f"Waveform shape: {trace.waveform.shape}")

Channel: CH1
Sample rate: 1000000 Sa/s
Waveform shape: (1000,)


## Using with a lambda get function

For simple cases where you don't need a subclass, you can pass a `get_cmd` callable
that returns the struct instance. This works because `StructParameter` inherits
the `get_cmd` support from `ParameterBase`.

In [13]:
@dataclass
class TemperatureReading:
    """Temperature sensor reading."""

    temperature: float
    humidity: float


def fake_read_sensor() -> TemperatureReading:
    return TemperatureReading(
        temperature=22.5 + rng.normal(0, 0.1),
        humidity=45.0 + rng.normal(0, 1.0),
    )


temp_sensor = StructParameter(
    "environment",
    struct_type=TemperatureReading,
    get_cmd=fake_read_sensor,
    field_labels={"temperature": "Temperature", "humidity": "Relative Humidity"},
    field_units={"temperature": "°C", "humidity": "%"},
)

print(temp_sensor.get())

TemperatureReading(temperature=22.56985663453715, humidity=45.10192588737266)


## Snapshot behaviour

By default, `StructParameter` sets `snapshot_value=False` so that the snapshot
does not call `get()` on the instrument during snapshotting. The snapshot still
includes metadata like the struct type name, field names, labels, and units.

In [14]:
snap = iv.snapshot()
for key in ("name", "struct_type_name", "names", "labels", "units"):
    print(f"  {key}: {snap[key]}")

  name: iv
  struct_type_name: IVResult
  names: ('voltage', 'current')
  labels: ('Voltage', 'Current')
  units: ('V', 'A')
